In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch


model_name = "LiquidAI/LFM2.5-350M-Base-sft-imdb/checkpoint-1200" 

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    torch_dtype="auto",
)
tokenizer = AutoTokenizer.from_pretrained(model_name)

print(f"Parameters: {model.num_parameters():,}")
print(f"Vocab size: {len(tokenizer)}")
print(f"Model size: ~{model.get_memory_footprint() / 2**30:.1f} GB")

In [ ]:
def finish_sentence(model, tokenizer, input_string, **sampling_params):
    inputs = tokenizer(input_string, return_tensors="pt").to("cuda")
    output = model.generate(
        **inputs,
        do_sample=True,
        max_new_tokens=128,
    )
    decoded = tokenizer.decode(output[0][1:])
    return decoded

In [ ]:
for _ in range(10):
    print(finish_sentence(model, tokenizer, "This movie was"))

In [ ]:
import datasets
import pandas as pd
from transformers import AutoModelForSequenceClassification, AutoTokenizer
import numpy as np
import matplotlib.pyplot as plt

def filter_dataset(filtered_ds, negative_sentiment=False):
    reward_model = AutoModelForSequenceClassification.from_pretrained("imdb_reward_model", device_map='cuda')
    reward_tokenizer = AutoTokenizer.from_pretrained("imdb_reward_model")
    
    def remove_prompt(example):
        prompt_len = len(example['prompt'])

        whitespace = " " if (example['chosen'][0] != ' ' and example['prompt'][-1]!=' ') else ""
        example['chosen'] = whitespace+example['chosen'][prompt_len:] 

        whitespace = " " if (example['rejected'][0] != ' ' and example['prompt'][-1]!=' ') else ""
        example['rejected'] = whitespace+example['rejected'][prompt_len:] 
        return example
    
    def swap_prefs(example):
        example['chosen'], example['rejected'] = example['rejected'], example['chosen']
        return example
    
    def patch_reward(example):
        inputs = reward_tokenizer([example['chosen'], example['rejected']], padding=True, return_tensors='pt').to('cuda')
        with torch.no_grad():
            reward = reward_model(**inputs).logits
            example['chosen_weight'] = example['chosen_reward'] #torch.exp(reward[0, 0])
            example['rejected_weight'] = example['rejected_reward'] #reward[1, 0]
        return example
    
    
    # filtered_ds = filtered_ds.map(patch_reward)
    filtered_ds = filtered_ds.filter(lambda x: x["chosen_reward"] > 0 > x["rejected_reward"]).shuffle(seed=42)
    filtered_ds = filtered_ds.map(remove_prompt)
    filtered_ds = filtered_ds.remove_columns(['chosen_reward', 'rejected_reward'])
    if negative_sentiment:
        filtered_ds = filtered_ds.map(swap_prefs)

    return filtered_ds.shuffle(seed=42)


train_dataset = filter_dataset(datasets.load_dataset("yuasosnin/imdb-dpo", split="train"), negative_sentiment=True)
eval_dataset =  filter_dataset(datasets.load_dataset("yuasosnin/imdb-dpo", split="test"), negative_sentiment=True)

In [ ]:
experiment_id = "./LiquidAI/LFM2.5-350M-sft-imdb-pspo-sentiment-b05-ls05"

In [ ]:
from trl import DPOConfig, DPOTrainer

max_seq_length = 1048
dpo_trainer = DPOTrainer(
    model=model,
    ref_model=None,
    processing_class=tokenizer,
    train_dataset = train_dataset,
    eval_dataset = eval_dataset,
    args = DPOConfig(
        loss_type="myipo",
        label_smoothing=0.5,
        beta = .5,
        num_train_epochs=2,
        per_device_train_batch_size=4,
        per_device_eval_batch_size=4,
        eval_accumulation_steps=1,
        gradient_accumulation_steps=2,
        learning_rate=2e-6,
        lr_scheduler_type="cosine",
        warmup_ratio=0.1,
        logging_steps=10,
        save_strategy="steps",
        eval_strategy='steps',
        adam_beta1=0.9,
        adam_beta2=0.999,
        save_steps=50,
        eval_steps=50,
        gradient_checkpointing=True,
        load_best_model_at_end=True,
        report_to='tensorboard',
        dataloader_persistent_workers=False,
        eval_on_start=True,
        max_length=512,
        output_dir=experiment_id,
        remove_unused_columns = False,
        ),
)



In [ ]:
dpo_trainer.train()

In [ ]:
dpo_trainer.save_model()
print(f"Model saved to: {experiment_id}")

In [ ]:
import torch 
torch.cuda.empty_cache()

In [ ]:
for _ in range(10):
    print(finish_sentence(model, tokenizer, "This movie was"))